# MLflow Integration - AutoRAG Mock Data with OpenShift Authentication

This notebook demonstrates MLflow SDK integration for **AutoRAG** (Documents RAG optimization pipeline), following the same parent-child pattern as AutoML.

**Prerequisites:**
```bash
oc login -u <username> -p '<password>' https://api.ai-eng-gpu.socc.p3.openshiftapps.com:443
export MLFLOW_WORKSPACE="ai-eng-cracow"
export MLFLOW_TRACKING_URI="https://rh-ai.apps.rosa.ai-eng-gpu.socc.p3.openshiftapps.com/mlflow"
export MLFLOW_TRACKING_TOKEN="$(oc whoami --show-token)"
```

Reference: `/architecture-decision-records/documentation/components/autorag/features/mlflow_integration.md`

## Imports and Setup

In [1]:
import os
import json
import subprocess
from pathlib import Path

import mlflow
from mlflow.tracking import MlflowClient

print(f"MLflow version: {mlflow.__version__}")
print(f"Python version: {subprocess.run(['python', '--version'], capture_output=True, text=True).stdout.strip()}")

MLflow version: 3.12.0
Python version: Python 3.12.13


## Configure MLflow (Following Working Demo Pattern)

Set environment variables and configure MLflow exactly like the working demo script.

In [2]:
# Step 1: Get authentication token (if not already set)
if not os.environ.get("MLFLOW_TRACKING_TOKEN"):
    try:
        token = subprocess.run(
            ["oc", "whoami", "--show-token"],
            capture_output=True,
            text=True,
            timeout=5,
            check=True
        ).stdout.strip()
        os.environ["MLFLOW_TRACKING_TOKEN"] = token
        print("✅ Token retrieved from 'oc whoami --show-token'")
    except Exception as e:
        print(f"❌ ERROR: Could not get token from oc CLI: {e}")
        print("   You MUST set MLFLOW_TRACKING_TOKEN manually")
        raise RuntimeError("MLFLOW_TRACKING_TOKEN is required for cluster access")

# Step 2: Set MLflow environment variables (keep them in environment!)
MLFLOW_WORKSPACE = os.environ.get("MLFLOW_WORKSPACE", "ai-eng-cracow")
MLFLOW_TRACKING_URI = os.environ.get(
    "MLFLOW_TRACKING_URI",
    "https://rh-ai.apps.rosa.ai-eng-gpu.socc.p3.openshiftapps.com/mlflow"
)

# Ensure environment variables are set (following working demo pattern)
os.environ["MLFLOW_WORKSPACE"] = MLFLOW_WORKSPACE
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI

# Step 3: Verify we have all required credentials
token = os.environ.get("MLFLOW_TRACKING_TOKEN")
if not token:
    raise RuntimeError("❌ MLFLOW_TRACKING_TOKEN is not set. Cannot proceed without authentication.")

print(f"\n{'='*70}")
print("MLFLOW CONFIGURATION VERIFICATION")
print(f"{'='*70}")
print(f"✓ Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"✓ Workspace: {MLFLOW_WORKSPACE}")
print(f"✓ Token: SET (length: {len(token)} chars)")
print(f"{'='*70}")

# Step 4: Configure MLflow client with explicit settings
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print("\n🔍 Testing cluster connectivity...")
import requests
try:
    # Test basic connectivity to the cluster
    response = requests.get(
        f"{MLFLOW_TRACKING_URI}/health",
        headers={"Authorization": f"Bearer {token}"},
        timeout=10
    )
    print(f"✓ Health check response: {response.status_code}")
    if response.status_code == 200:
        print("✓ Cluster is reachable and responding")
    else:
        print(f"⚠️  Unexpected status code: {response.status_code}")
        print(f"   Response: {response.text[:200]}")
except Exception as e:
    print(f"❌ Cluster connectivity test failed: {e}")
    raise

print("\n✅ Configuration complete - ready to use REMOTE cluster")
print("❌ Local fallback is DISABLED - all operations will use the cluster")

✓ Health check response: 200
✓ Cluster is reachable and responding

✅ Configuration complete - ready to use REMOTE cluster
❌ Local fallback is DISABLED - all operations will use the cluster


## Verify Cluster Connectivity

Run these diagnostics to ensure cluster access is working before proceeding.

In [3]:
# Diagnostic: Test MLflow API connectivity directly
print("="*70)
print("CLUSTER CONNECTIVITY DIAGNOSTICS")
print("="*70)

token = os.environ.get("MLFLOW_TRACKING_TOKEN")
tracking_uri = os.environ.get("MLFLOW_TRACKING_URI")
workspace = os.environ.get("MLFLOW_WORKSPACE")

print(f"\n1. Environment Variables:")
print(f"   MLFLOW_TRACKING_URI: {tracking_uri}")
print(f"   MLFLOW_WORKSPACE: {workspace}")
print(f"   MLFLOW_TRACKING_TOKEN: {'SET' if token else 'NOT SET'}")

print(f"\n2. Testing API endpoints...")

# Test 1: Server info endpoint (this is what's failing)
print(f"\n   Test A: /api/2.0/mlflow/server-info")
try:
    response = requests.get(
        f"{tracking_uri}/api/2.0/mlflow/server-info",
        headers={"Authorization": f"Bearer {token}"},
        timeout=10
    )
    print(f"   Status: {response.status_code}")
    if response.status_code == 200:
        try:
            data = response.json()
            print(f"   Response: {data}")
        except json.JSONDecodeError:
            print(f"   Response (not JSON): {response.text[:200]}")
    else:
        print(f"   Error response: {response.text[:200]}")
except Exception as e:
    print(f"   ERROR: {e}")

# Test 2: List experiments endpoint with workspace
print(f"\n   Test B: /api/2.0/mlflow/experiments/list?workspace={workspace}")
try:
    response = requests.get(
        f"{tracking_uri}/api/2.0/mlflow/experiments/list",
        headers={"Authorization": f"Bearer {token}"},
        params={"workspace": workspace},
        timeout=10
    )
    print(f"   Status: {response.status_code}")
    if response.status_code == 200:
        try:
            data = response.json()
            print(f"   Found {len(data.get('experiments', []))} experiments")
        except json.JSONDecodeError:
            print(f"   Response (not JSON): {response.text[:200]}")
    else:
        print(f"   Error response: {response.text[:200]}")
except Exception as e:
    print(f"   ERROR: {e}")

# Test 3: Verify token with oc command
print(f"\n3. Verifying OpenShift authentication...")
try:
    result = subprocess.run(
        ["oc", "whoami"],
        capture_output=True,
        text=True,
        timeout=5
    )
    if result.returncode == 0:
        print(f"   ✓ Logged in as: {result.stdout.strip()}")
    else:
        print(f"   ✗ Not logged in: {result.stderr.strip()}")
except Exception as e:
    print(f"   ERROR: {e}")

print(f"\n{'='*70}")
print("If any test failed, fix the issue before proceeding")
print("="*70)

   ✓ Logged in as: ai-eng-gpu-adm

If any test failed, fix the issue before proceeding


In [4]:
# Set experiment (with workspace probe bypass)
EXPERIMENT_NAME = "AutoRAG-Documents-RAG-Optimization"

print(f"\n{'='*70}")
print("SETTING MLFLOW EXPERIMENT")
print(f"{'='*70}")

# Verify configuration is still set
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Workspace (env): {os.environ.get('MLFLOW_WORKSPACE')}")
print(f"Token set: {'Yes' if os.environ.get('MLFLOW_TRACKING_TOKEN') else 'NO - ERROR!'}")

# WORKAROUND: Server-info endpoint returns non-JSON, breaking workspace probe
# But API endpoints require workspace parameter
# Solution: Monkey-patch the RestStore to bypass workspace probe
print(f"\n⚠️  Applying workspace probe bypass")
print(f"   Reason: server-info endpoint broken but workspace is required")

from mlflow.store.tracking.rest_store import RestStore

# Save original method
original_probe = RestStore._probe_workspace_support

# Replace with method that always returns True (workspace supported)
def patched_probe(self):
    return True

RestStore._probe_workspace_support = patched_probe
print(f"   ✓ Workspace probe bypassed - will always send workspace parameter")

try:
    print(f"\nAttempting to set experiment: {EXPERIMENT_NAME}")
    mlflow.set_experiment(EXPERIMENT_NAME)
    print(f"✅ Experiment set successfully: {EXPERIMENT_NAME}")
    
    # Get experiment info to verify
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if experiment:
        print(f"\n✓ Experiment ID: {experiment.experiment_id}")
        print(f"✓ Artifact Location: {experiment.artifact_location}")
        print(f"✓ Lifecycle Stage: {experiment.lifecycle_stage}")
    else:
        print("⚠️  Warning: Experiment was set but could not retrieve details")
        
except Exception as e:
    print(f"\n❌ Error setting experiment: {e}")
    print(f"   Error type: {type(e).__name__}")
    import traceback
    traceback.print_exc()
    raise

print(f"{'='*70}")

✅ Experiment set successfully: AutoRAG-Documents-RAG-Optimization

✓ Experiment ID: 1
✓ Artifact Location: /mlflow/artifacts/workspaces/ai-eng-cracow/1
✓ Lifecycle Stage: active


In [5]:
# KFP Pipeline Context
kfp_context = {
    "pipeline_name": "documents-rag-optimization-pipeline",
    "run_id": "run-rag-abc123-def456",
    "run_name": "autorag-optimization-2026-05-08-091500",
    "kfp_version": "2.8.0"
}

# Pipeline Parameters (Parent run params per design doc)
pipeline_params = {
    "optimization_metric": "answer_correctness",
    "optimization_max_rag_patterns": 5,
    "llama_stack_vector_io_provider_id": "milvus-provider",
    "image": "quay.io/opendatahub/autorag:v1.0.0",
    "ai4rag_version": "0.5.0",
    "dataset_uri": "s3://my-bucket/datasets/financial_qa.json"
}

# Mock RAG patterns - aligned with design doc (all 5 metrics + confidence intervals)
rag_patterns = {
    "pattern_01": {
        "name": "pattern_01",
        "iteration": 1,
        "max_combinations": 24,
        "duration_seconds": 234.56,
        "settings": {
            "vector_store_binding": {
                "provider_id": "milvus-provider",
                "provider_type": "remote::milvus",
                "vector_store_id": "vs_coll_pattern_01",
                "vector_store_name": "AutoRAG pattern_01 vector store"
            },
            "chunking": {
                "method": "recursive",
                "chunk_size": 512,
                "chunk_overlap": 50
            },
            "embedding": {
                "model_id": "text-embedding-3-small",
                "distance_metric": "cosine",
                "embedding_params": {"embedding_dimension": 768}
            },
            "retrieval": {
                "method": "simple",
                "number_of_chunks": 5,
                "search_mode": "hybrid",
                "ranker_strategy": "rrf",
                "ranker_k": 60,
                "ranker_alpha": 0.5
            },
            "generation": {
                "model_id": "meta-llama/Llama-3.1-8B-Instruct",
                "context_template_text": "{document}",
                "user_message_text": "\n\nContext:\n{reference_documents}:\n\nQuestion: {question}.",
                "system_message_text": "Please answer the question using the provided context."
            },
            "responses_template": {
                "model": "meta-llama/Llama-3.1-8B-Instruct",
                "stream": False,
                "store": True,
                "instructions": "Please answer the question using only information found in file_search results. If the question is unanswerable, say you cannot answer.",
                "tools": [
                    {
                        "type": "file_search",
                        "vector_store_ids": ["vs_coll_pattern_01"],
                        "max_num_results": 5
                    }
                ]
            }
        },
        "scores": {
            "faithfulness": {"mean": 0.91, "ci_low": 0.88, "ci_high": 0.94},
            "answer_correctness": {"mean": 0.8947, "ci_low": 0.86, "ci_high": 0.93},
            "context_precision": {"mean": 0.87, "ci_low": 0.84, "ci_high": 0.90},
            "context_recall": {"mean": 0.89, "ci_low": 0.86, "ci_high": 0.92},
            "answer_relevancy": {"mean": 0.90, "ci_low": 0.87, "ci_high": 0.93}
        },
        "final_score": 0.8947,
        "evaluation_results": [
            {
                "question": "What is the refund policy?",
                "answer": "The refund policy allows for a full refund within 30 days of purchase.",
                "answer_contexts": [
                    {
                        "text": "Refund policy: Customers may request a full refund within 30 days of purchase.",
                        "document_id": "policy.pdf"
                    }
                ],
                "scores": {
                    "faithfulness": 0.95,
                    "answer_correctness": 0.92,
                    "context_precision": 0.88,
                    "context_recall": 0.91,
                    "answer_relevancy": 0.94
                }
            },
            {
                "question": "What payment methods are accepted?",
                "answer": "We accept credit cards, debit cards, and PayPal.",
                "answer_contexts": [
                    {
                        "text": "Payment methods: Visa, Mastercard, American Express, and PayPal.",
                        "document_id": "payment_info.pdf"
                    }
                ],
                "scores": {
                    "faithfulness": 0.87,
                    "answer_correctness": 0.87,
                    "context_precision": 0.86,
                    "context_recall": 0.87,
                    "answer_relevancy": 0.86
                }
            }
        ]
    },
    "pattern_02": {
        "name": "pattern_02",
        "iteration": 2,
        "max_combinations": 24,
        "duration_seconds": 198.34,
        "settings": {
            "vector_store_binding": {
                "provider_id": "qdrant-provider",
                "provider_type": "remote::qdrant",
                "vector_store_id": "vs_coll_pattern_02",
                "vector_store_name": "AutoRAG pattern_02 vector store"
            },
            "chunking": {
                "method": "semantic",
                "chunk_size": 768,
                "chunk_overlap": 100
            },
            "embedding": {
                "model_id": "text-embedding-ada-002",
                "distance_metric": "cosine",
                "embedding_params": {"embedding_dimension": 1536}
            },
            "retrieval": {
                "method": "simple",
                "number_of_chunks": 10,
                "search_mode": "semantic",
                "ranker_strategy": "reciprocal_rank_fusion",
                "ranker_k": 50,
                "ranker_alpha": 0.6
            },
            "generation": {
                "model_id": "mistralai/Mixtral-8x7B-Instruct-v0.1",
                "context_template_text": "{document}",
                "user_message_text": "Context:\n{reference_documents}\n\nQuestion: {question}",
                "system_message_text": "You are a helpful assistant. Answer based on the context provided."
            },
            "responses_template": {
                "model": "mistralai/Mixtral-8x7B-Instruct-v0.1",
                "stream": False,
                "store": True,
                "instructions": "Answer questions using file_search results. Be concise and accurate.",
                "tools": [
                    {
                        "type": "file_search",
                        "vector_store_ids": ["vs_coll_pattern_02"],
                        "max_num_results": 10
                    }
                ]
            }
        },
        "scores": {
            "faithfulness": {"mean": 0.89, "ci_low": 0.85, "ci_high": 0.93},
            "answer_correctness": {"mean": 0.8723, "ci_low": 0.83, "ci_high": 0.91},
            "context_precision": {"mean": 0.85, "ci_low": 0.81, "ci_high": 0.89},
            "context_recall": {"mean": 0.87, "ci_low": 0.83, "ci_high": 0.91},
            "answer_relevancy": {"mean": 0.88, "ci_low": 0.84, "ci_high": 0.92}
        },
        "final_score": 0.8723,
        "evaluation_results": [
            {
                "question": "What is the refund policy?",
                "answer": "Refunds are available within 30 days.",
                "answer_contexts": [
                    {
                        "text": "Refund: 30 days.",
                        "document_id": "policy.pdf"
                    }
                ],
                "scores": {
                    "faithfulness": 0.92,
                    "answer_correctness": 0.88,
                    "context_precision": 0.87,
                    "context_recall": 0.89,
                    "answer_relevancy": 0.90
                }
            },
            {
                "question": "What payment methods are accepted?",
                "answer": "Credit cards and PayPal are accepted.",
                "answer_contexts": [
                    {
                        "text": "Payment: credit cards, PayPal.",
                        "document_id": "payment_info.pdf"
                    }
                ],
                "scores": {
                    "faithfulness": 0.86,
                    "answer_correctness": 0.86,
                    "context_precision": 0.83,
                    "context_recall": 0.85,
                    "answer_relevancy": 0.86
                }
            }
        ]
    },
    "pattern_03": {
        "name": "pattern_03",
        "iteration": 3,
        "max_combinations": 24,
        "duration_seconds": 187.92,
        "settings": {
            "vector_store_binding": {
                "provider_id": "milvus-provider",
                "provider_type": "inline::milvus",
                "vector_store_id": "vs_coll_pattern_03",
                "vector_store_name": "AutoRAG pattern_03 vector store"
            },
            "chunking": {
                "method": "fixed",
                "chunk_size": 256,
                "chunk_overlap": 25
            },
            "embedding": {
                "model_id": "text-embedding-3-large",
                "distance_metric": "cosine",
                "embedding_params": {"embedding_dimension": 3072}
            },
            "retrieval": {
                "method": "simple",
                "number_of_chunks": 3,
                "search_mode": "dense",
                "ranker_strategy": "max_marginal_relevance",
                "ranker_k": 40,
                "ranker_alpha": 0.3
            },
            "generation": {
                "model_id": "ibm/granite-3.1-8b-instruct",
                "context_template_text": "{document}",
                "user_message_text": "Based on: {reference_documents}\nAnswer: {question}",
                "system_message_text": "Provide accurate answers based on the given context."
            },
            "responses_template": {
                "model": "ibm/granite-3.1-8b-instruct",
                "stream": False,
                "store": True,
                "instructions": "Use file_search to answer. Stay factual and cite sources when possible.",
                "tools": [
                    {
                        "type": "file_search",
                        "vector_store_ids": ["vs_coll_pattern_03"],
                        "max_num_results": 3
                    }
                ]
            }
        },
        "scores": {
            "faithfulness": {"mean": 0.87, "ci_low": 0.83, "ci_high": 0.91},
            "answer_correctness": {"mean": 0.8612, "ci_low": 0.82, "ci_high": 0.90},
            "context_precision": {"mean": 0.84, "ci_low": 0.80, "ci_high": 0.88},
            "context_recall": {"mean": 0.86, "ci_low": 0.82, "ci_high": 0.90},
            "answer_relevancy": {"mean": 0.87, "ci_low": 0.83, "ci_high": 0.91}
        },
        "final_score": 0.8612,
        "evaluation_results": [
            {
                "question": "What is the refund policy?",
                "answer": "30-day refund.",
                "answer_contexts": [
                    {
                        "text": "30-day policy.",
                        "document_id": "policy.pdf"
                    }
                ],
                "scores": {
                    "faithfulness": 0.90,
                    "answer_correctness": 0.86,
                    "context_precision": 0.85,
                    "context_recall": 0.87,
                    "answer_relevancy": 0.88
                }
            },
            {
                "question": "What payment methods are accepted?",
                "answer": "Cards and PayPal.",
                "answer_contexts": [
                    {
                        "text": "Payment: cards, PayPal.",
                        "document_id": "payment_info.pdf"
                    }
                ],
                "scores": {
                    "faithfulness": 0.84,
                    "answer_correctness": 0.86,
                    "context_precision": 0.83,
                    "context_recall": 0.85,
                    "answer_relevancy": 0.86
                }
            }
        ]
    }
}

pattern_names = list(rag_patterns.keys())
patterns_artifact_path = "/mnt/artifacts/rag-patterns-2026-05-08"

print(f"✅ Mock RAG pattern data prepared for {len(pattern_names)} patterns")
print(f"   - All 5 evaluation metrics included: faithfulness, answer_correctness,")
print(f"     context_precision, context_recall, answer_relevancy")
print(f"   - RHOAI 3.5+ structure: responses_template + vector_store_binding")
print(f"   - Confidence intervals for all metrics")

✅ Mock RAG pattern data prepared for 3 patterns
   - All 5 evaluation metrics included: faithfulness, answer_correctness,
     context_precision, context_recall, answer_relevancy
   - RHOAI 3.5+ structure: responses_template + vector_store_binding
   - Confidence intervals for all metrics


## Log AutoRAG Results to MLflow

In [6]:
def log_autorag_to_mlflow(
    rag_patterns: dict,
    pattern_names: list,
    pipeline_params: dict,
    kfp_context: dict,
    patterns_artifact_path: str,
    workspace: str,
    log_artifacts: bool = False
):
    """
    Log AutoRAG results to MLflow using PARENT-CHILD RUN STRUCTURE.
    
    Following industry standard pattern (Optuna, FLAML, PySpark):
    - Parent run = KFP pipeline execution (RAG optimization pipeline)
    - Child runs = individual RAG patterns tested (nested)
    
    Aligned with: /architecture-decision-records/documentation/components/autorag/features/mlflow_integration.md
    
    Args:
        log_artifacts: If True, attempts to log artifact files. 
                      Set to False when using remote MLflow with local artifact storage.
    """
    
    print("\n" + "="*60)
    print("LOGGING AUTORAG TO MLFLOW - PARENT-CHILD PATTERN")
    print(f"Pipeline: {kfp_context['run_name']}")
    print(f"RAG Patterns: {len(pattern_names)}")
    print("="*60)
    
    # ================================================================
    # PARENT RUN: Represents the KFP pipeline execution
    # ================================================================
    with mlflow.start_run(run_name=kfp_context['run_name']) as parent_run:
        
        parent_run_id = parent_run.info.run_id
        print(f"\n✅ Parent run created: {parent_run_id}")
        print(f"   Represents: KFP RAG optimization pipeline execution")
        
        # ============================================================
        # PARENT: Tags (per design doc)
        # ============================================================
        print("\n📌 Logging pipeline-level TAGS to parent...")
        parent_tags = {
            "pipeline_name": kfp_context['pipeline_name'],
            "kfp_run_name": kfp_context['run_name'],
            "kfp_run_id": kfp_context['run_id'],
            "kfp_version": kfp_context['kfp_version'],
            "image": pipeline_params['image'],
            "ai4rag_version": pipeline_params['ai4rag_version']
        }
        mlflow.set_tags(parent_tags)
        
        # ============================================================
        # PARENT: Parameters (per design doc)
        # ============================================================
        print("⚙️  Logging pipeline-level PARAMS to parent...")
        
        # Calculate aggregate statistics
        final_scores = [p['final_score'] for p in rag_patterns.values()]
        durations = [p['duration_seconds'] for p in rag_patterns.values()]
        
        parent_params = {
            "optimization_metric": pipeline_params['optimization_metric'],
            "optimization_max_rag_patterns": str(pipeline_params['optimization_max_rag_patterns']),
            "llama_stack_vector_io_provider_id": pipeline_params['llama_stack_vector_io_provider_id'],
            "dataset_uri": pipeline_params['dataset_uri']
        }
        mlflow.log_params(parent_params)
        
        # ============================================================
        # PARENT: Metrics (aggregate statistics per design doc)
        # ============================================================
        print("📊 Logging pipeline-level METRICS to parent...")
        
        # Per design doc: best_score, worst_score, mean_score, num_patterns_evaluated, total_duration_seconds
        parent_metrics = {
            "best_score": max(final_scores),
            "worst_score": min(final_scores),
            "mean_score": sum(final_scores) / len(final_scores),
            "num_patterns_evaluated": len(pattern_names),
            "total_duration_seconds": sum(durations)
        }
        mlflow.log_metrics(parent_metrics)
        
        print(f"   - best_score: {parent_metrics['best_score']:.4f}")
        print(f"   - worst_score: {parent_metrics['worst_score']:.4f}")
        print(f"   - mean_score: {parent_metrics['mean_score']:.4f}")
        print(f"   - num_patterns_evaluated: {parent_metrics['num_patterns_evaluated']}")
        print(f"   - total_duration_seconds: {parent_metrics['total_duration_seconds']:.2f}s")
        
        # ============================================================
        # PARENT: Artifacts (shared pipeline artifacts)
        # ============================================================
        if log_artifacts:
            try:
                print("📄 Logging pipeline-level ARTIFACTS to parent...")
                
                # Create pattern comparison HTML
                comparison_html = """<html>
<head><title>RAG Pattern Comparison</title></head>
<body>
    <h1>RAG Pattern Evaluation Results</h1>
    <table border="1">
        <tr><th>Rank</th><th>Pattern</th><th>Score</th><th>Duration</th><th>Chunking</th><th>Embedding</th><th>LLM</th></tr>
"""
                for rank, name in enumerate(sorted(rag_patterns.keys(), 
                                                   key=lambda x: rag_patterns[x]['final_score'], 
                                                   reverse=True), 1):
                    p = rag_patterns[name]
                    comparison_html += f"""        <tr>
            <td>{rank}</td>
            <td>{name}</td>
            <td>{p['final_score']:.4f}</td>
            <td>{p['duration_seconds']:.2f}s</td>
            <td>{p['settings']['chunking']['method']}</td>
            <td>{p['settings']['embedding']['model_id']}</td>
            <td>{p['settings']['generation']['model_id']}</td>
        </tr>
"""
                comparison_html += """    </table>
</body>
</html>"""
                
                comparison_path = Path("pattern_comparison.html")
                comparison_path.write_text(comparison_html)
                mlflow.log_artifact(str(comparison_path), artifact_path="reports")
                comparison_path.unlink()
                print("   - pattern_comparison.html")
            except Exception as e:
                print(f"    ⚠️  Could not log artifacts: {e}")
        
        # ============================================================
        # CHILD RUNS: One nested run per RAG pattern
        # ============================================================
        print(f"\n👶 Creating {len(pattern_names)} CHILD RUNS (nested)...")
        print("─"*60)
        
        child_run_ids = []
        
        for i, pattern_name in enumerate(pattern_names, 1):
            pattern = rag_patterns[pattern_name]
            
            # Create nested child run for this pattern
            with mlflow.start_run(run_name=pattern_name, nested=True) as child_run:
                
                child_run_id = child_run.info.run_id
                child_run_ids.append(child_run_id)
                
                # --------------------------------------------------------
                # CHILD: Tags (pattern identity)
                # --------------------------------------------------------
                mlflow.set_tags({
                    "pattern_name": pattern['name'],
                    "iteration": str(pattern['iteration']),
                    "chunking_method": pattern['settings']['chunking']['method'],
                    "embedding_model": pattern['settings']['embedding']['model_id'],
                    "generation_model": pattern['settings']['generation']['model_id'],
                    "vector_store_provider": pattern['settings']['vector_store_binding']['provider_type']
                })
                
                # --------------------------------------------------------
                # CHILD: Parameters (pattern-specific settings)
                # --------------------------------------------------------
                mlflow.log_params({
                    "iteration": pattern['iteration'],
                    "max_combinations": pattern['max_combinations'],
                    "duration_seconds": pattern['duration_seconds'],
                    "chunk_size": pattern['settings']['chunking']['chunk_size'],
                    "chunk_overlap": pattern['settings']['chunking']['chunk_overlap'],
                    "chunking_method": pattern['settings']['chunking']['method'],
                    "embedding_model_id": pattern['settings']['embedding']['model_id'],
                    "embedding_dimension": pattern['settings']['embedding']['embedding_params']['embedding_dimension'],
                    "distance_metric": pattern['settings']['embedding']['distance_metric'],
                    "retrieval_method": pattern['settings']['retrieval']['method'],
                    "number_of_chunks": pattern['settings']['retrieval']['number_of_chunks'],
                    "search_mode": pattern['settings']['retrieval']['search_mode'],
                    "ranker_strategy": pattern['settings']['retrieval']['ranker_strategy'],
                    "generation_model_id": pattern['settings']['generation']['model_id'],
                    "vector_store_id": pattern['settings']['vector_store_binding']['vector_store_id'],
                    "vector_store_provider": pattern['settings']['vector_store_binding']['provider_type'],
                    # Artifact pointers
                    "pattern_artifact_path": f"{patterns_artifact_path}/{pattern_name}"
                })
                
                # --------------------------------------------------------
                # CHILD: Metrics (pattern performance - all 5 metrics with CIs)
                # --------------------------------------------------------
                metrics_to_log = {}
                
                # Log all 5 RAG evaluation metrics with confidence intervals
                for metric_name in ['faithfulness', 'answer_correctness', 'context_precision', 
                                   'context_recall', 'answer_relevancy']:
                    metric_data = pattern['scores'][metric_name]
                    metrics_to_log[f"{metric_name}_mean"] = metric_data['mean']
                    metrics_to_log[f"{metric_name}_ci_low"] = metric_data['ci_low']
                    metrics_to_log[f"{metric_name}_ci_high"] = metric_data['ci_high']
                
                # Add final score
                metrics_to_log['final_score'] = pattern['final_score']
                
                mlflow.log_metrics(metrics_to_log)
                
                print(f"   [{i}/{len(pattern_names)}] {pattern_name:<15} | Score: {pattern['final_score']:.4f} | ID: {child_run_id[:8]}...")
        
        print("\n" + "="*60)
        print("✅ LOGGING COMPLETED")
        print("="*60)
        print(f"Parent Run ID: {parent_run_id}")
        print(f"Child Runs: {len(child_run_ids)}")
        print(f"Best Score: {parent_metrics['best_score']:.4f}")
        print(f"Total Duration: {parent_metrics['total_duration_seconds']:.2f}s")
        print(f"\n💡 Parent-child structure follows industry standard (Optuna/FLAML)")
        
        # Return tracking info
        experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
        return {
            "tracking_enabled": True,
            "mlflow_tracking_uri": mlflow.get_tracking_uri(),
            "mlflow_experiment_id": experiment.experiment_id,
            "mlflow_experiment_name": experiment.name,
            "mlflow_parent_run_id": parent_run_id,
            "mlflow_child_run_ids": child_run_ids,
            "mlflow_workspace": workspace,
            "total_patterns": len(child_run_ids),
            "best_score": parent_metrics['best_score'],
            "worst_score": parent_metrics['worst_score'],
            "mean_score": parent_metrics['mean_score'],
            "total_duration_seconds": parent_metrics['total_duration_seconds'],
            "kfp_run_id": kfp_context['run_id'],
            "pipeline_name": kfp_context['pipeline_name'],
            "pattern_names": pattern_names
        }

# Execute logging with parent-child structure
tracking_info = log_autorag_to_mlflow(
    rag_patterns=rag_patterns,
    pattern_names=pattern_names,
    pipeline_params=pipeline_params,
    kfp_context=kfp_context,
    patterns_artifact_path=patterns_artifact_path,
    workspace=MLFLOW_WORKSPACE,
    log_artifacts=False  # Set to True only if artifact storage is writable
)

🏃 View run autorag-optimization-2026-05-08-091500 at: https://rh-ai.apps.rosa.ai-eng-gpu.socc.p3.openshiftapps.com/mlflow/#/experiments/1/runs/5ef4eb4e7be84cfc96e0ff0eda42cb22?workspace=ai-eng-cracow
🧪 View experiment at: https://rh-ai.apps.rosa.ai-eng-gpu.socc.p3.openshiftapps.com/mlflow/#/experiments/1?workspace=ai-eng-cracow


## Query Logged Data

In [7]:
client = MlflowClient()

print("="*60)
print("QUERYING PARENT-CHILD RUN STRUCTURE")
print("="*60)

parent_run_id = tracking_info['mlflow_parent_run_id']
experiment_id = tracking_info['mlflow_experiment_id']

# ============================================================
# Query Parent Run
# ============================================================
parent_run = client.get_run(parent_run_id)

print(f"\n🎯 PARENT RUN (Pipeline Execution):")
print(f"{'─'*60}")
print(f"Run ID: {parent_run.info.run_id}")
print(f"Run Name: {parent_run.info.run_name}")
print(f"Status: {parent_run.info.status}")

print(f"\n  📌 Tags:")
for k, v in sorted(parent_run.data.tags.items()):
    if not k.startswith('mlflow.'):
        print(f"    {k}: {v}")

print(f"\n  ⚙️  Parameters:")
for k, v in sorted(parent_run.data.params.items()):
    print(f"    {k}: {v}")

print(f"\n  📊 Metrics:")
for k, v in sorted(parent_run.data.metrics.items()):
    print(f"    {k}: {v}")

# ============================================================
# Query Child Runs
# ============================================================
child_runs = client.search_runs(
    experiment_ids=[experiment_id],
    filter_string=f"tags.mlflow.parentRunId = '{parent_run_id}'"
)

print(f"\n👶 CHILD RUNS ({len(child_runs)} RAG patterns):")
print(f"{'─'*60}")

# Sort by final_score descending
child_runs_sorted = sorted(child_runs, key=lambda r: r.data.metrics.get('final_score', 0), reverse=True)

for i, child in enumerate(child_runs_sorted, 1):
    pattern_name = child.data.tags.get('pattern_name', child.info.run_name)
    final_score = child.data.metrics.get('final_score', 0)
    chunking = child.data.params.get('chunking_method', 'N/A')
    embedding = child.data.params.get('embedding_model_id', 'N/A')
    generation = child.data.params.get('generation_model_id', 'N/A')
    
    print(f"\n  [{i}] {pattern_name}")
    print(f"      Run ID: {child.info.run_id[:16]}...")
    print(f"      Final Score: {final_score:.4f}")
    print(f"      Chunking: {chunking} | Embedding: {embedding}")
    print(f"      Generation: {generation}")
    print(f"      Metrics: ", end="")
    
    # Display all 5 RAG metrics (mean values)
    rag_metrics = ['faithfulness', 'answer_correctness', 'context_precision', 
                   'context_recall', 'answer_relevancy']
    metric_strs = [f"{m}={child.data.metrics.get(f'{m}_mean', 0):.4f}" 
                   for m in rag_metrics if f'{m}_mean' in child.data.metrics]
    print(", ".join(metric_strs))

print(f"\n{'='*60}")
print(f"✅ Query completed - Parent-child hierarchy verified")
print(f"{'='*60}")
print(f"\n💡 Structure: 1 parent (pipeline) + {len(child_runs)} children (RAG patterns)")


👶 CHILD RUNS (3 RAG patterns):
────────────────────────────────────────────────────────────

  [1] pattern_01
      Run ID: 36e787a622df4aa7...
      Final Score: 0.8947
      Chunking: recursive | Embedding: text-embedding-3-small
      Generation: meta-llama/Llama-3.1-8B-Instruct
      Metrics: faithfulness=0.9100, answer_correctness=0.8947, context_precision=0.8700, context_recall=0.8900, answer_relevancy=0.9000

  [2] pattern_02
      Run ID: 454eb50b1d5c4fe4...
      Final Score: 0.8723
      Chunking: semantic | Embedding: text-embedding-ada-002
      Generation: mistralai/Mixtral-8x7B-Instruct-v0.1
      Metrics: faithfulness=0.8900, answer_correctness=0.8723, context_precision=0.8500, context_recall=0.8700, answer_relevancy=0.8800

  [3] pattern_03
      Run ID: 23f4d01c7cf245de...
      Final Score: 0.8612
      Chunking: fixed | Embedding: text-embedding-3-large
      Generation: ibm/granite-3.1-8b-instruct
      Metrics: faithfulness=0.8700, answer_correctness=0.8612, conte

---

## MLflow Tracing (RHOAI 3.6+)

MLflow Tracing provides detailed observability for RAG applications using OpenTelemetry Protocol (OTLP).

**Key Concepts:**
- One trace per evaluation question (not per pattern)
- Trace hierarchy: CHAIN → RETRIEVER → LLM
- Multiple patterns tested against the same questions = multiple CHAIN spans per trace
- Traces linked to child runs via `run_id` attribute

**UI Navigation:**
1. Go to MLflow UI → Experiments → "AutoRAG-Documents-RAG-Optimization"
2. Click on parent run: `autorag-optimization-2026-05-08-091500`
3. Click "Traces" tab
4. Select a trace (e.g., "Question: What is the refund policy?")
5. View the trace tree showing all 3 patterns tested (pattern_01, pattern_02, pattern_03)

In [8]:
import mlflow
import mlflow.tracing

print("\n" + "="*80)
print("MLFLOW TRACING DEMONSTRATION - RAG EVALUATION")
print("="*80)

# Check MLflow version and tracing availability
print(f"\n🔍 Checking MLflow Tracing configuration...")
print(f"   MLflow version: {mlflow.__version__}")

# Try to import SpanType - it may not exist in all versions
try:
    from mlflow.entities import SpanType
    print(f"   SpanType imported from mlflow.entities")
    HAS_SPAN_TYPE = True
except ImportError:
    try:
        # Try alternate import path
        from mlflow.tracing.constant import SpanType
        print(f"   SpanType imported from mlflow.tracing.constant")
        HAS_SPAN_TYPE = True
    except ImportError:
        print(f"   ⚠️  SpanType not available - will use string types")
        HAS_SPAN_TYPE = False
        # Define fallback constants
        class SpanType:
            CHAIN = "CHAIN"
            RETRIEVER = "RETRIEVER"
            LLM = "LLM"

print(f"   Tracing module available: {hasattr(mlflow, 'tracing')}")
print(f"   start_span available: {hasattr(mlflow, 'start_span')}")

# Get all unique questions from the evaluation results
questions_set = set()
for pattern in rag_patterns.values():
    for eval_result in pattern['evaluation_results']:
        questions_set.add(eval_result['question'])

questions = sorted(list(questions_set))

print(f"\n📋 Found {len(questions)} unique evaluation questions")
print(f"📋 {len(pattern_names)} patterns will be traced")
print(f"📋 Total traces to create: {len(questions) * len(pattern_names)}")
print(f"   (1 trace per pattern per question)")

# ============================================================================
# IMPORTANT: Create traces within each CHILD RUN context
# This creates a proper link between traces and the child runs (patterns)
# ============================================================================

experiment_id = tracking_info['mlflow_experiment_id']

print(f"\n🔗 Creating traces within child run contexts for proper linkage")

try:
    trace_count = 0

    # ========================================================================
    # Create traces within each child run (pattern)
    # This ensures traces are linked to the correct pattern run
    # ========================================================================

    for pattern_idx, pattern_name in enumerate(pattern_names, 1):
        pattern = rag_patterns[pattern_name]
        child_run_id = tracking_info['mlflow_child_run_ids'][pattern_idx - 1]

        print(f"\n{'─'*80}")
        print(f"Pattern [{pattern_idx}/{len(pattern_names)}]: {pattern_name}")
        print(f"  Child run ID: {child_run_id}")
        print(f"{'─'*80}")

        # Reopen the CHILD run to create traces within it
        with mlflow.start_run(run_id=child_run_id, experiment_id=experiment_id):

            print(f"  ✓ Child run context activated: {mlflow.active_run().info.run_id}")

            for question_idx, question in enumerate(questions, 1):

                # Find this question's evaluation result for this pattern
                eval_result = None
                for result in pattern['evaluation_results']:
                    if result['question'] == question:
                        eval_result = result
                        break

                if eval_result is None:
                    print(f"   ⚠️  No eval result for question {question_idx}")
                    continue

                try:
                    # Create a trace for this pattern answering this question
                    # Since we're in the child run context, the trace is automatically linked
                    with mlflow.start_span(
                        name=f"{pattern_name} - {question}",
                        span_type=SpanType.CHAIN,
                        attributes={
                            "pattern_name": pattern_name,
                            "question": question,
                            "iteration": pattern['iteration'],
                            "source": "AutoRAG",
                            "chunking_method": pattern['settings']['chunking']['method'],
                            "embedding_model": pattern['settings']['embedding']['model_id'],
                            "generation_model": pattern['settings']['generation']['model_id'],
                            # Add all 5 RAG metrics
                            "faithfulness": eval_result['scores']['faithfulness'],
                            "answer_correctness": eval_result['scores']['answer_correctness'],
                            "context_precision": eval_result['scores']['context_precision'],
                            "context_recall": eval_result['scores']['context_recall'],
                            "answer_relevancy": eval_result['scores']['answer_relevancy']
                        }
                    ) as rag_pipeline_span:

                        trace_count += 1

                        # Set the REQUEST (inputs) for the RAG pipeline
                        rag_pipeline_span.set_inputs({
                            "question": question,
                            "pattern_name": pattern_name,
                            "pattern_settings": {
                                "chunking": {
                                    "method": pattern['settings']['chunking']['method'],
                                    "chunk_size": pattern['settings']['chunking']['chunk_size'],
                                    "chunk_overlap": pattern['settings']['chunking']['chunk_overlap']
                                },
                                "embedding": {
                                    "model_id": pattern['settings']['embedding']['model_id'],
                                    "dimension": pattern['settings']['embedding']['embedding_params']['embedding_dimension']
                                },
                                "retrieval": {
                                    "method": pattern['settings']['retrieval']['method'],
                                    "number_of_chunks": pattern['settings']['retrieval']['number_of_chunks'],
                                    "search_mode": pattern['settings']['retrieval']['search_mode']
                                },
                                "generation": {
                                    "model_id": pattern['settings']['generation']['model_id']
                                }
                            }
                        })

                        # RETRIEVER span - document retrieval phase
                        with mlflow.start_span(
                            name="retrieve_documents",
                            span_type=SpanType.RETRIEVER,
                            attributes={
                                "vector_store_id": pattern['settings']['vector_store_binding']['vector_store_id'],
                                "search_mode": pattern['settings']['retrieval']['search_mode'],
                                "number_of_chunks": pattern['settings']['retrieval']['number_of_chunks'],
                                "ranker_strategy": pattern['settings']['retrieval']['ranker_strategy'],
                                "ranker_k": pattern['settings']['retrieval']['ranker_k'],
                                "ranker_alpha": pattern['settings']['retrieval']['ranker_alpha']
                            }
                        ) as retriever_span:

                            # Set retriever inputs
                            retriever_span.set_inputs({
                                "question": question,
                                "vector_store_id": pattern['settings']['vector_store_binding']['vector_store_id'],
                                "number_of_chunks": pattern['settings']['retrieval']['number_of_chunks']
                            })

                            # Set retrieved documents as outputs
                            retriever_span.set_outputs({
                                "documents": eval_result['answer_contexts'],
                                "num_documents": len(eval_result['answer_contexts'])
                            })

                        # LLM span - answer generation phase
                        # Build the actual prompt that would be sent to the LLM
                        context_text = "\n\n".join([
                            f"Document {i+1}: {ctx['text']}"
                            for i, ctx in enumerate(eval_result['answer_contexts'])
                        ])

                        # Format the user message using the template
                        user_message_template = pattern['settings']['generation']['user_message_text']
                        user_message = user_message_template.replace('{reference_documents}', context_text).replace('{question}', question)

                        system_message = pattern['settings']['generation']['system_message_text']

                        # Build the complete prompt as it would be sent to the LLM
                        complete_prompt = f"System: {system_message}\n\nUser: {user_message}"

                        with mlflow.start_span(
                            name="generate_answer",
                            span_type=SpanType.LLM,
                            attributes={
                                "model": pattern['settings']['generation']['model_id'],
                                "system_message": system_message,
                                "user_message_template": pattern['settings']['generation']['user_message_text']
                            }
                        ) as llm_span:

                            # Set LLM inputs with the actual formatted prompt
                            llm_span.set_inputs({
                                "prompt": complete_prompt,
                                "messages": [
                                    {"role": "system", "content": system_message},
                                    {"role": "user", "content": user_message}
                                ],
                                "model": pattern['settings']['generation']['model_id'],
                                "temperature": 0.0
                            })

                            # Set LLM output
                            llm_span.set_outputs({
                                "answer": eval_result['answer']
                            })

                        # Set the RESPONSE (outputs) for the RAG pipeline
                        rag_pipeline_span.set_outputs({
                            "answer": eval_result['answer'],
                            "retrieved_documents": eval_result['answer_contexts'],
                            "scores": eval_result['scores']
                        })

                    print(f"   ✓ Trace {trace_count}: Q{question_idx} | "
                          f"Score: {eval_result['scores']['answer_correctness']:.4f} | "
                          f"Answer: {eval_result['answer'][:40]}...")

                except Exception as e:
                    print(f"   ❌ Error creating trace: {e}")
                    import traceback
                    traceback.print_exc()

    print(f"\n{'='*80}")
    print(f"✅ TRACING COMPLETED")
    print(f"{'='*80}")
    print(f"Total traces created: {trace_count}")
    print(f"Expected: {len(pattern_names)} patterns × {len(questions)} questions = {len(pattern_names) * len(questions)}")
    print(f"\n💡 Each trace is linked to its corresponding child run (pattern)")
    print(f"💡 Trace structure: CHAIN (RAG pipeline) → RETRIEVER → LLM")
    print(f"💡 Request field: question + pattern settings")
    print(f"💡 Response field: answer + retrieved docs + scores")
    print(f"💡 Prompt field: complete formatted prompt sent to LLM")
    print(f"💡 Source attribute: AutoRAG")
    print(f"\n🔗 View traces in MLflow UI:")
    print(f"   1. Go to: {mlflow.get_tracking_uri()}")
    print(f"   2. Navigate to experiment: {EXPERIMENT_NAME}")
    print(f"   3. Option A - View from parent run:")
    print(f"      • Click parent run: {kfp_context['run_name']}")
    print(f"      • Traces from all child runs should be visible")
    print(f"   4. Option B - View from child run (pattern):")
    print(f"      • Click a child run: pattern_01, pattern_02, or pattern_03")
    print(f"      • Click 'Traces' tab")
    print(f"      • See {len(questions)} traces for that specific pattern")
    print(f"{'='*80}")

except Exception as e:
    print(f"\n❌ ERROR during tracing: {e}")
    import traceback
    traceback.print_exc()
    print(f"\n⚠️  Tracing may not be supported on this MLflow server")
    print(f"   Check MLflow server version and tracing configuration")

[Trace(trace_id=tr-cd34ef729b24a5fc781ac5e577de12df), Trace(trace_id=tr-0689151938bfab446422a50d9a7b866b), Trace(trace_id=tr-6067cecea0ea626b3adf0b8f87696833), Trace(trace_id=tr-30e8f64311bae6544920f8329a42b912), Trace(trace_id=tr-0aa3744029580927c0b62636f7e42bae)]

## Verify Traces Were Created

The tracing demonstration above created the following structure:

**Trace Summary:**
- **Total traces**: 6 (one per pattern per question)
  - 2 traces for pattern_01 (one per question)
  - 2 traces for pattern_02 (one per question)  
  - 2 traces for pattern_03 (one per question)
- **Each trace contains**: 2 sub-spans (RETRIEVER + LLM)

**Trace Structure:**
```
Trace: "pattern_01 - What is the refund policy?"
├── retrieve_documents (RETRIEVER)
└── generate_answer (LLM)

Trace: "pattern_01 - What payment methods are accepted?"
├── retrieve_documents (RETRIEVER)
└── generate_answer (LLM)

Trace: "pattern_02 - What is the refund policy?"
├── retrieve_documents (RETRIEVER)
└── generate_answer (LLM)

Trace: "pattern_02 - What payment methods are accepted?"
├── retrieve_documents (RETRIEVER)
└── generate_answer (LLM)

Trace: "pattern_03 - What is the refund policy?"
├── retrieve_documents (RETRIEVER)
└── generate_answer (LLM)

Trace: "pattern_03 - What payment methods are accepted?"
├── retrieve_documents (RETRIEVER)
└── generate_answer (LLM)
```

**What Each Trace Contains:**
- ✅ Root CHAIN span: Full RAG pipeline for one pattern answering one question
  - Pattern name, question, linked child run_id
  - All 5 RAG metrics as attributes
  - Final answer
- ✅ RETRIEVER span: Document retrieval
  - Vector store details, search settings
  - Retrieved documents as outputs
- ✅ LLM span: Answer generation
  - Model, prompts
  - Input (question + context) and output (answer)

**How to Compare Patterns:**
In MLflow UI, you can filter/sort traces by:
- Pattern name (pattern_01, pattern_02, pattern_03)
- Question
- Metrics (answer_correctness, faithfulness, etc.)

This allows easy side-by-side comparison of how different patterns performed on the same question.

In [9]:
# Verify traces were uploaded to MLflow server
print("="*80)
print("VERIFYING TRACES IN MLFLOW")
print("="*80)

parent_run_id = tracking_info['mlflow_parent_run_id']

try:
    # Check if we can query traces via MLflow API
    print(f"\n🔍 Checking for traces associated with run: {parent_run_id}")
    
    # Try to get trace info using MLflow client
    client = MlflowClient()
    
    # Get the run to see if it has trace information
    run = client.get_run(parent_run_id)
    
    print(f"✓ Retrieved run: {run.info.run_name}")
    print(f"  Status: {run.info.status}")
    
    # Check if there's a traces endpoint we can query
    print(f"\n🔍 Attempting to query traces via API...")
    
    import requests
    token = os.environ.get("MLFLOW_TRACKING_TOKEN")
    tracking_uri = os.environ.get("MLFLOW_TRACKING_URI")
    workspace = os.environ.get("MLFLOW_WORKSPACE")
    
    # Try the traces API endpoint
    traces_url = f"{tracking_uri}/api/2.0/mlflow/traces/search"
    
    response = requests.post(
        traces_url,
        headers={"Authorization": f"Bearer {token}"},
        json={
            "experiment_ids": [tracking_info['mlflow_experiment_id']],
            "max_results": 100
        },
        timeout=10
    )
    
    print(f"  Traces API status: {response.status_code}")
    
    if response.status_code == 200:
        traces_data = response.json()
        traces = traces_data.get('traces', [])
        print(f"  ✓ Found {len(traces)} traces in experiment")
        
        if len(traces) > 0:
            print(f"\n  Trace details:")
            for i, trace in enumerate(traces[:5], 1):  # Show first 5
                trace_id = trace.get('request_id', 'unknown')
                trace_name = trace.get('request_metadata', {}).get('mlflow.traceName', 'unnamed')
                print(f"    [{i}] {trace_name} (ID: {trace_id})")
        else:
            print(f"\n  ⚠️  No traces found in experiment")
            print(f"     Possible reasons:")
            print(f"     1. MLflow server version doesn't support tracing")
            print(f"     2. Traces weren't uploaded correctly")
            print(f"     3. Tracing is disabled on the server")
    
    elif response.status_code == 404:
        print(f"  ⚠️  Traces API endpoint not found (404)")
        print(f"     This MLflow server may not support tracing")
        print(f"     Tracing was added in MLflow 2.14+")
        
    else:
        print(f"  ❌ Error querying traces: {response.status_code}")
        print(f"     Response: {response.text[:200]}")
    
    print(f"\n{'='*80}")
    print(f"TRACE VERIFICATION COMPLETE")
    print(f"{'='*80}")
    
    print(f"\n📝 Summary:")
    print(f"  • Parent run exists: ✓")
    print(f"  • Traces API available: {'✓' if response.status_code == 200 else '✗'}")
    print(f"  • Traces found: {len(traces) if response.status_code == 200 else 0}")
    
    if response.status_code != 200 or len(traces) == 0:
        print(f"\n💡 Note: If traces aren't visible, this may be expected.")
        print(f"   MLflow Tracing requires:")
        print(f"   • MLflow server version 2.14 or higher")
        print(f"   • RHOAI 3.6+ for full tracing support")
        print(f"   • Tracing enabled on the MLflow server")
        print(f"\n   The parent-child run structure and metrics are still logged correctly.")
    
except Exception as e:
    print(f"\n❌ Error verifying traces: {e}")
    import traceback
    traceback.print_exc()
    
    print(f"\n💡 This is expected if the MLflow server doesn't support tracing.")
    print(f"   The parent-child run structure is still logged successfully.")

print(f"{'='*80}")

Trace(trace_id=tr-8b2973a0a7f33b6c73ad206cbf6cd3d0)

---

## Summary: What Was Logged to MLflow

This notebook demonstrates the complete AutoRAG MLflow integration with simplified pattern names.

### 📊 Successfully Logged Components

#### 1. Parent-Child Run Structure ✅
- **1 Parent Run**: `autorag-optimization-2026-05-08-091500`
  - Represents the KFP pipeline execution
  - Contains aggregate metrics: best_score, worst_score, mean_score
  - Contains pipeline parameters: optimization_metric, dataset_uri, etc.
  
- **3 Child Runs**: `pattern_01`, `pattern_02`, `pattern_03`
  - Each represents one RAG pattern tested
  - Contains pattern-specific parameters: chunk_size, embedding_model, etc.
  - Contains all 5 RAG metrics with confidence intervals:
    - faithfulness (mean, ci_low, ci_high)
    - answer_correctness (mean, ci_low, ci_high)
    - context_precision (mean, ci_low, ci_high)
    - context_recall (mean, ci_low, ci_high)
    - answer_relevancy (mean, ci_low, ci_high)

#### 2. MLflow Tracing (RHOAI 3.6+) - Server Dependent ⚠️
**Note**: Tracing visibility depends on MLflow server capabilities.

**If tracing is supported on your server**, the following traces were created:
- **6 Traces**: One per pattern per question
  - 2 traces for pattern_01
  - 2 traces for pattern_02
  - 2 traces for pattern_03
  
- **Each trace contains**:
  - Root CHAIN span: Complete RAG pipeline execution
  - RETRIEVER span: Document retrieval with vector store details
  - LLM span: Answer generation with inputs/outputs
  - All 5 RAG metrics as attributes on the root span

**Trace names format**: `pattern_01 - What is the refund policy?`

This structure allows you to:
- Compare how different patterns (pattern_01, pattern_02, pattern_03) answered the same question
- Filter traces by pattern name or question
- See the complete RAG pipeline breakdown for each evaluation

**Check Cell 17 to verify if traces were uploaded to the server.**

**If traces exist, see Cell 18 for detailed navigation instructions and Cell 19 for direct links.**

### 🎯 Pattern Names
All patterns use simplified naming:
- ✅ `pattern_01` (was: pattern_001_recursive_llama3_milvus)
- ✅ `pattern_02` (was: pattern_002_semantic_mistral_qdrant)
- ✅ `pattern_03` (was: pattern_003_fixed_granite_milvus)

### 🔧 Workarounds Applied
- **Workspace probe bypass**: Monkey-patched `RestStore._probe_workspace_support()` to avoid broken server-info endpoint
- **Cluster-only**: No local fallback - all operations use remote MLflow server
- **Token security**: Token value is never printed, only status and length

### 📍 How to View in MLflow UI

**Option 1: Use Direct Links (Cell 19)**
- Run Cell 19 to get clickable links
- Click the parent run link
- Look for "Traces" tab

**Option 2: Manual Navigation (Cell 18)**
- Follow step-by-step instructions in Cell 18
- Navigate: Experiments → AutoRAG-Documents-RAG-Optimization → Parent run → Traces tab

**What You'll See:**
1. **Runs** (always available):
   - Parent run with aggregate metrics
   - 3 child runs with individual pattern results
   
2. **Traces** (if server supports it):
   - 6 traces showing individual RAG executions
   - Each trace: pattern_XX - question
   - Full RAG pipeline breakdown: CHAIN → RETRIEVER → LLM
   
3. **Metrics Comparison**:
   - Compare all 5 RAG metrics across patterns
   - View confidence intervals
   - Filter by pattern or question

### 🎓 Key Learning
This notebook demonstrates the **recommended MLflow integration pattern** for AutoRAG:
- Parent run = pipeline execution
- Child runs = individual patterns
- Traces = individual pattern executions (one per pattern per evaluation question)
- All metrics with confidence intervals
- Tracing for detailed observability (when available)

---